# Lecture 19 · How to make nice figures — 학생용

### ▶ Colab 에서 시작하기 (처음 한 번)
1. **colab.research.google.com** → `파일` → `노트북 업로드` → 이 `.ipynb` 파일 선택
2. 맨 위 메뉴 `런타임` → `모두 실행` 은 **하지 않는다** — 위에서부터 셀을 하나씩 `Shift + Enter`
3. 🔒 셀 (라이브러리 · 데이터) 은 실행만 하면 된다. 데이터는 인터넷에서 바로 읽으니 **파일을 올릴 필요가 없다**
4. 런타임이 끊기면 (`연결 끊김`) 🔒 셀부터 다시 실행한다

### 오늘 하는 방식
- 단계마다 **새로 들어가는 1 ~ 3 줄만** 직접 친다. 이미 친 코드는 다시 치지 않는다.
- 단계마다 빈 셀이 있다 → **바로 위 내 셀을 통째로 복사** (셀 안에서 `Ctrl+A` → `Ctrl+C`) → 빈 셀에 붙여넣기 → 안내된 줄을 넣고 실행.
- 안내 표시 — ➕ 추가 (어느 줄 다음에) · ✏️ 바꾸기 (이전 → 이후) · ➖ 지우기
- 🔮 치기 전에 **그림이 어떻게 바뀔지 먼저 예측**해 본다.
- 🏁 절 끝의 **체크포인트** 셀 — 막혔으면 실행해서 따라잡는다 (코드는 접혀 있다).
- 🔒 표시 셀 — 데이터 · 준비 코드. **실행만** 한다.
- `Tab` 자동완성 · `Shift+Tab` 함수 설명 · 짧은 이름 `mfc` `ms` `ls` 를 쓰면 덜 친다.




### 💡 그림을 만들 때의 일반 원칙

| # | 원칙 | 코드 |
|---|---|---|
| 1 | **템플릿에서 시작한다** — 매번 처음부터 쓰지 않는다 | 이 노트북의 1 절 |
| 2 | **정사각형 · 작게** — 논문 한 단 폭에 맞춘다 | `figsize=(5, 5)` · 2 단 편집이면 3.5 |
| 3 | **글자는 한 번에 14** · 일러스트레이터에서 고칠 수 있게 | `rcParams['font.size']=14` · `pdf.fonttype=42` |
| 4 | **데이터는 점, 모형은 선** · 마커 속은 흰색 | `marker` · `mfc='w'` · `linestyle='none'` |
| 5 | **순서가 있으면 순차형 색** · 직접 고른 색도 좋다 | `sns.color_palette('rocket', n)` · Seshadri 색 |
| 6 | **눈금은 네 변 · 안쪽 · 크게**, 숫자는 왼쪽 · 아래만 | `tick_params(direction='in', length=10/5, ...)` |
| 7 | **범위와 눈금은 직접 정한다** — 자동값은 출발점일 뿐 | `xlim` · `ylim` · `xticks(np.arange(...))` |
| 8 | **같은 정보는 한 번만** — 패널끼리 공유하는 축 이름 · 숫자는 하나 | `labelleft=False` · `plt.text` |
| 9 | **데이터를 줄이거나 바꿨으면 그림 설명에 쓴다** | `df.iloc[::10]` → "1/10 로 표시" |
| 10 | **범례는 그림과 같은 순서 · 데이터를 가리지 않게** | `reversed(...)` · `bbox_to_anchor` |
| 11 | **글자로 직접 표시** — 범례를 찾아보게 하지 않는다 | `plt.text` · `axvline` |
| 12 | **임의 단위면 숫자를 지운다** | `labelleft=False` |
| 13 | **저장은 고해상도 · 여백 자르기 · 벡터** | `dpi=300, bbox_inches='tight'` · `.pdf` `.eps` |
| 14 | **완벽하게 안 되면 벡터로 저장해 손으로 마무리** | `.eps` → Illustrator |

In [ ]:
#@title 🔒 라이브러리 (실행만)
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.ticker import MultipleLocator
import seaborn as sns

In [ ]:
#@title 🔒 글자 크기 · 데이터 위치 · 색 (실행만)
mpl.rcParams['font.size'] = 14         # 💡3 글자 크기를 한 번에
mpl.rcParams['pdf.fonttype'] = 42      # 💡3 PDF 글자를 글자로 저장 (일러스트레이터에서 편집)


# 데이터 위치 — 아무것도 올리지 않아도 된다 (수업 GitHub 에서 바로 읽는다)
BASE = 'https://raw.githubusercontent.com/gwanyeong/my_projects/master/chem-data-science/lecture19/data/'
# BASE = 'data/'     # 직접 올린 data 폴더를 쓸 때만 이 줄로 바꾼다
seshadri = ['#c3121e', '#0348a1', '#ffb01c', '#027608', '#0193b0', '#9c5300', '#949c01', '#7104b5']
#            0sangre,   1neptune,  2pumpkin,  3clover,   4denim,    5cocoa,    6cumin,    7berry

---
## 1 · 템플릿 그림 

입자 크기 (µm) 에 따른 세 물질의 기여도. 가로축이 1 ~ 10000 으로 넓게 퍼진 데이터.

**이 절의 의미**
- Sparks 는 무엇을 그리든 이 파일을 열고 시작한다 (영상 0:53) → **템플릿 = 모든 그림의 출발점**
- 영상도 코드를 처음부터 치지 않고 **덩어리 (chunk) 별로** 읽어 나간다 — 오늘 방식과 같음
- 흐름: 라이브러리 → 데이터 → 색 → 그리기 → 범위 · 눈금 → 이름 · 범례 → 저장
- 핵심 메시지: **기본값은 출발점일 뿐 — 아쉬운 점을 말하고, 옵션 하나씩, 다시 본다**


In [ ]:
#@title 🔒 데이터 (실행만)
df = pd.read_csv(BASE + 'generic_single_plot.csv')
x,  y  = df['x'].dropna(),  df['y'].dropna()      # 열마다 길이가 달라 NaN 을 버린다
x2, y2 = df['x2'].dropna(), df['y2'].dropna()
x3, y3 = df['x3'].dropna(), df['y3'].dropna()
df.head()

#### 1-1 · 기본값 그대로
아무 옵션 없이 그린다. 이것이 출발점이다. **무엇이 아쉬운지 먼저 말해 보자.**

**🎯 강의 포인트**
- 옵션 없이 그린 그림 = 출발점. **먼저 무엇이 아쉬운지 말하게 한다**
- 학생 답 유도: 가로로 긺 · 왼쪽에 뭉침 · 점이 안 보임 · 눈금 바깥 · 이름 없음
- 영상 3:00 — "나쁘지 않지만 더 할 수 있다"


**이번 단계 — 새 셀에 직접 친다** (주석 `# …` 은 안 쳐도 된다)
```python
plt.plot(x3, y3)
plt.plot(x2, y2)
plt.plot(x, y)
plt.show()
```

In [ ]:
# 1-1 · 직접 친다


#### 1-2 · 정사각형 그림판
💡2 가로로 긴 기본 그림 (6.4 × 4.8) 대신 5 × 5 인치.

**🎯 강의 포인트**
- `figsize` 단위 = 인치. 기본 (6.4, 4.8) → 5 × 5
- Sparks 는 **정사각형**을 선호 (영상 2:21)
- 2 단 편집 논문이면 한 단 폭 3.5 인치 → 처음부터 그 크기로 그리면 글자 크기가 그대로 유지
- `plt.figure` 는 반드시 그리기 **전에** — 그림판을 먼저 깐다


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — **맨 위에**
```python
fig = plt.figure(1, figsize=(5, 5))      # ← 바꿔 보기 : (6.4, 4.8) · (3.5, 3.5)
```

In [ ]:
# 1-2 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-3 · 가로축을 로그로
가로 데이터가 1 ~ 10000 이라 선형 축에서는 왼쪽에 다 뭉친다. **축의 종류는 데이터 범위가 정한다.**

**🎯 강의 포인트**
- x 가 1 ~ 10⁴ → 선형 축에서는 왼쪽에 다 뭉침
- **축의 종류는 데이터 범위가 정한다**
- `semilogx` = x 만 로그 · `semilogy` = y 만 · `loglog` = 둘 다
- 로그 축은 0 · 음수를 못 그림 (에러 없이 빠짐) — 주의


🔮 **예측** — 가로축만 로그로 바꾸면 세 곡선은 어떻게 보일까?

**이번 단계** (주석 `# …` 은 안 쳐도 된다)

✏️ 바꾸기 — 이 줄(들)을
```python
plt.plot(x3, y3)
plt.plot(x2, y2)
plt.plot(x, y)
```

　　　→ 이렇게
```python
plt.semilogx(x3, y3)                       # ← 바꿔 보기 : plt.plot 으로 되돌려 비교
plt.semilogx(x2, y2)
plt.semilogx(x, y)
```

In [ ]:
# 1-3 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-4 · 마커 · 흰 속 · 이름표
💡4 측정한 점이 보이게. 세 물질은 마커 **모양**도 다르게 해서 흑백에서도 구별되게.

**🎯 강의 포인트**
- 측정값은 **점이 보여야** 한다 → `marker`
- `mfc='w'` (흰 속) — Sparks 의 습관 · 겹친 점도 구별
- 세 데이터의 **마커 모양을 다르게** (^ · s · o) → 흑백 인쇄 · 색각 이상에도 구별
- `label` 은 이름표만 붙임 — 범례 상자는 나중에 `legend()` 로


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

✏️ 줄 안에 끼워 넣기 — `⟦ ⟧` 안만 새로 친다
```python
plt.semilogx(x3, y3⟦, linestyle='-', marker='^', mfc='w', markersize=8, label='ammonium perchlorate'⟧)
plt.semilogx(x2, y2⟦, linestyle='-', marker='s', mfc='w', markersize=8, label='iron oxide'⟧)
plt.semilogx(x, y⟦, linestyle='-', marker='o', mfc='w', markersize=8, label='aluminum'⟧)
```

In [ ]:
# 1-4 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-5 · 색
💡 `rocket` 에서 3 색. 연속 팔레트의 양 끝과 가운데를 준다.

바꿔 보기 → `'crest'`, `'mako'`, 또는 `seshadri[0:3]`

**🎯 강의 포인트**
- Sparks 가 가장 좋아하는 `rocket` 팔레트 (영상 1:50)
- `color_palette('rocket', 3)` → 연속 색띠의 양 끝과 가운데에서 3 색
- 직접 고른 색도 가능 — 박사후 지도교수 Seshadri 의 hex 색 목록 (영상 2:02)
- 순서가 있는 데이터 → 순차형 · 순서가 없으면 범주형 (Part I)


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — **맨 위에**
```python
colors_ = sns.color_palette('rocket', 3)   # ← 바꿔 보기 : 'crest' · 'mako' · seshadri[:3]
```

✏️ 줄 안에 끼워 넣기 — `⟦ ⟧` 안만 새로 친다
```python
plt.semilogx(x3, y3, linestyle='-', marker='^', mfc='w', markersize=8, label='ammonium perchlorate'⟦, color=colors_[2]⟧)
plt.semilogx(x2, y2, linestyle='-', marker='s', mfc='w', markersize=8, label='iron oxide'⟦, color=colors_[1]⟧)
plt.semilogx(x, y, linestyle='-', marker='o', mfc='w', markersize=8, label='aluminum'⟦, color=colors_[0]⟧)
```

In [ ]:
# 1-5 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-6 · 범위를 직접
💡7 가로 1 ~ 10⁴, 세로 −0.5 ~ 16. 자동 범위는 데이터 끝에 딱 붙거나 애매한 곳에서 끝난다.

**🎯 강의 포인트**
- 자동 범위는 데이터 끝에 딱 붙거나 애매한 곳에서 끝남
- 가로 1 ~ 10⁴ 로 늘리고, 세로 −0.5 ~ 16 (0 이 축에 붙지 않게 여유)
- **자동값은 출발점 · 범위는 직접 정한다** (영상 3:18)


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `plt.semilogx(x,  y,  linestyle='-', marker='o', mfc='w', mar` **다음 줄에**
```python
plt.xlim([1, 1e4])
plt.ylim([-0.5, 16])                       # ← 바꿔 보기 : 이 두 줄을 지우고 비교
```

In [ ]:
# 1-6 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-7 · 눈금을 안쪽 · 네 변으로
💡6 기본 눈금은 바깥 · 아래 · 왼쪽에만. 위와 오른쪽에도 켜면 그림이 상자로 닫힌다.

**🎯 강의 포인트**
- 기본: 눈금이 바깥 · 아래 · 왼쪽에만
- `direction='in'` + `top · right=True` → **네 변 모두 안쪽** — 그림이 상자로 닫힘
- Sparks 의 취향이자 물리화학 논문 그림의 일반적인 모양 (영상 3:13)


🔮 **예측** — 눈금을 '안쪽 · 네 변' 으로 — 숫자도 위 · 오른쪽에 생길까?

**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `plt.ylim([-0.5, 16])                       # ← 바꿔 보기 : 이 두 줄` **다음 줄에**
```python
plt.tick_params(direction='in', top=True, right=True)   # ← 바꿔 보기 : 'out' · 'inout'
```

In [ ]:
# 1-7 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-8 · 작은 눈금 · 눈금 길이
기본 눈금은 너무 짧아 잘 안 보인다. 큰 눈금 10, 작은 눈금 5.

**🎯 강의 포인트**
- 기본 눈금은 너무 짧아 잘 안 보임 (영상 5:11)
- 큰 눈금 10 · 작은 눈금 5 (포인트) · `which='major' / 'minor'` 로 따로 지정
- `minorticks_on()` — 작은 눈금 켜기


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

✏️ 바꾸기 — 이 줄(들)을
```python
plt.tick_params(direction='in', top=True, right=True)   # ← 바꿔 보기 : 'out' · 'inout'
```

　　　→ 이렇게
```python
plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
```

In [ ]:
# 1-8 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-9 · 숫자는 왼쪽 · 아래만
눈금은 네 변, **숫자**는 두 변. 위 · 오른쪽에도 숫자를 켜면 어떻게 되는지 `True` 로 바꿔 보자.

**🎯 강의 포인트**
- **눈금은 네 변, 숫자는 두 변 (왼쪽 · 아래)**
- 위 · 오른쪽에도 숫자를 켤 수는 있지만 "그러지 말라" (영상 4:06)
- 같은 정보를 두 번 보여 줄 이유가 없음 → 3 절 '같은 정보는 한 번만' 의 예고


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `plt.tick_params(direction='in', which='minor', length=5,  bo` **다음 줄에**
```python
plt.tick_params(labelbottom=True, labeltop=False,       # ← 바꿔 보기 : labeltop=True
                labelleft=True, labelright=False)       #             labelright=True
plt.tick_params(labelsize=14)
```

In [ ]:
# 1-9 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-10 · 세로 눈금 간격
자동 눈금은 2 마다 — 빽빽하다. 4 마다로.

`np.arange(0, 16, 4)` 를 쓰면 16 이 빠진다 (끝 값 미포함). 그래서 **16.1**.

**🎯 강의 포인트**
- 자동 눈금은 2 마다 → 빽빽 · 4 마다로 줄임
- `np.arange(0, 16, 4)` → 0 · 4 · 8 · 12 — **끝 값 16 은 빠짐**
- 그래서 `16.1` — 16 을 넣기 위한 요령 (영상 4:45)
- 기본 그래프 시간의 `np.arange(0, 101, 10)` 과 같은 규칙


🔮 **예측** — `np.arange(0, 16, 4)` 를 쓰면 16 눈금이 있을까?

**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `plt.tick_params(labelsize=14)` **다음 줄에**
```python
yticks = np.arange(0, 16.1, 4)             # ← 바꿔 보기 : np.arange(0, 16, 4) → 16 이 사라진다
plt.yticks(yticks)
```

In [ ]:
# 1-10 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-11 · 축 이름 · 범례
단위까지. `$\mu$` 는 수학 모드의 그리스 문자 µ. 백슬래시가 있으니 문자열 앞에 `r`.

**🎯 강의 포인트**
- 축 이름에는 **물리량 + 단위**
- `$\mu$` = 수학 모드의 그리스 문자 µ · 백슬래시가 있으면 문자열 앞에 `r`
- `legend()` 를 불러야 이름표가 상자로 나타남


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `plt.yticks(yticks)` **다음 줄에**
```python
plt.xlabel(r'particle radii ($\mu m$)')
plt.ylabel('percent contribution')
plt.legend()                               # ← 바꿔 보기 : loc='upper left' · fontsize=11 · frameon=False
```

In [ ]:
# 1-11 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 1-12 · 저장
💡13 `dpi=300` 고해상도, `bbox_inches='tight'` 여백 자르기. 완성 — **이 셀을 복사해서 다음 그림을 시작한다** (💡1).

**🎯 강의 포인트**
- `dpi=300` — 논문 그림 해상도 · `bbox_inches='tight'` — 빈 여백을 잘라 냄 (영상 6:24)
- `.pdf` · `.eps` 로 저장하면 벡터 → 일러스트레이터에서 편집 가능
- 완성 — **이 셀을 복사해서 다음 그림을 시작한다** (템플릿의 의미)
- 정리: 매번 바뀐 건 한두 줄뿐


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `plt.legend()                               # ← 바꿔 보기 : loc='` **다음 줄에**
```python
plt.savefig('generic_plot.png', dpi=300, bbox_inches='tight')   # ← 바꿔 보기 : .pdf (벡터)
```

In [ ]:
# 1-12 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


In [ ]:
#@title 🏁 체크포인트 · 1 절 완성 코드 (막혔으면 실행)
colors_ = sns.color_palette('rocket', 3)   # ← 바꿔 보기 : 'crest' · 'mako' · seshadri[:3]

fig = plt.figure(1, figsize=(5, 5))      # ← 바꿔 보기 : (6.4, 4.8) · (3.5, 3.5)

plt.semilogx(x3, y3, linestyle='-', marker='^', mfc='w', markersize=8, label='ammonium perchlorate', color=colors_[2])
plt.semilogx(x2, y2, linestyle='-', marker='s', mfc='w', markersize=8, label='iron oxide',           color=colors_[1])
plt.semilogx(x,  y,  linestyle='-', marker='o', mfc='w', markersize=8, label='aluminum',             color=colors_[0])

plt.xlim([1, 1e4])
plt.ylim([-0.5, 16])                       # ← 바꿔 보기 : 이 두 줄을 지우고 비교

plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
# ← 바꿔 보기 : length 를 4 / 2 (기본 근처) · 20 / 10

plt.tick_params(labelbottom=True, labeltop=False,       # ← 바꿔 보기 : labeltop=True
                labelleft=True, labelright=False)       #             labelright=True
plt.tick_params(labelsize=14)

yticks = np.arange(0, 16.1, 4)             # ← 바꿔 보기 : np.arange(0, 16, 4) → 16 이 사라진다
plt.yticks(yticks)

plt.xlabel(r'particle radii ($\mu m$)')
plt.ylabel('percent contribution')
plt.legend()                               # ← 바꿔 보기 : loc='upper left' · fontsize=11 · frameon=False

plt.savefig('generic_plot.png', dpi=300, bbox_inches='tight')   # ← 바꿔 보기 : .pdf (벡터)

plt.show()

---
## 2 · 누적 그림 

1998 ~ 2015 년 코발트 생산 점유율 (%),

**이 절의 의미**
- 데이터: 1998 ~ 2015 세계 코발트 생산 점유율 (%) · 22 개 나라 — 해마다 합이 100 %
- 합이 100 % 인 구성비의 시간 변화 → **누적 그림 (stackplot)** 이 맞는 형태
- 메시지: 2003 년 제2차 콩고 전쟁 종료 후 **DR 콩고 14 → 50 %**, 1998 년 1 위 **잠비아 34 → 4 %**
- 화학 연결: 코발트 = 리튬이온 배터리 양극재 (LiCoO₂ · NMC) 의 핵심 원소 → 한 나라 쏠림 = 공급망 위험
- 영상 7:15 — 학생이 숫자를 손으로 옮겨 적음 → "그래서 파이썬을 배운다"
- 관건: 범주가 22 개 → **색 · 범례 · 직접 표시**를 어떻게 하나


In [ ]:
#@title 🔒 데이터 (실행만)
countries =[ 'DR of the Congo','Australia', 'Botswana', 'Brazil',
            'Canada', 'China', 'Cuba', 'Finland', 'Indonesia', 
            'Kazakhstan', 'Madagascar', 'Morocco', 'New Caledonia',
            'Norway', 'Papua New Guinea', 'Phillippines', 'Russia',
            'South Africa', 'United States', 'Vietnam', 'Zambia',
            'Zimbabwe']
year = [1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015]

y1=[14.16, 21.41, 29.02, 26.79, 27.97, 27.98, 33.50, 37.58, 38.71, 34.46, 40.43, 49.88, 54.55, 53.15, 49.52, 49.12, 50.00, 50.00]
y2=[9.35, 12.54, 14.78, 14.06, 13.03, 11.34, 9.29, 7.04, 7.33, 6.42, 5.99, 5.42, 3.50, 3.47, 5.59, 5.62, 4.90, 4.76]
y3=[0.95, 1.01, 0.81, 0.73, 0.52, 0.56, 0.37, 0.50, 0.43, 0.33, 0.42, 0.43, 0.25, 0.13, 0.19, 0.22, 0.16, 0.25]
y4 =[1.13, 2.14, 2.37, 2.24, 2.11, 2.46, 2.32, 2.15, 1.57, 3.70, 3.29, 2.59, 2.85, 3.26, 2.76, 3.07, 3.14, 3.02]
y5=[16.60, 16.28, 13.98, 11.89, 9.86, 8.18, 8.39, 8.85, 10.16, 11.79, 11.21, 4.89, 4.21, 6.16, 6.36, 6.29, 5.66, 5.48]
y6=[0.11, 0.76, 0.24, 0.33, 1.92, 1.32, 2.09, 3.22, 2.63, 8.28, 8.30, 7.48, 5.80, 6.13, 7.14, 6.32, 6.31, 6.11]
y7=[7.55, 7.76, 7.53, 7.65, 6.59, 6.19, 5.89, 7.36, 8.00, 6.16, 5.01, 5.74, 4.36, 4.59, 4.67, 3.68, 3.03, 3.41]
y8=[0, 0, 0, 0, 0, 0.19, 0.17, 0.15, 0.14, 0.16, 0.13, 0.03, 0.13, 0.45, 0.60, 0.66, 0.63, 0.35]
y9=[0, 0, 0, 0, 0, 0, 2.8, 2.45, 2.29, 2.17, 1.63, 1.50, 1.45, 1.44, 1.62, 1.49, 1.07, 1.03]
y10=[0.85, 0.92, 0.79, 0.67, 0.57,0,0,0,0,0,0,0,0,0,0,0,0,0]
y11=[0,0,0,0,0,0,0,0,0,0,0,0,0.15, 0.45, 0.60, 1.93, 2.54, 2.94]
y12=[0.81, 2.64, 2.55, 2.77, 2.78, 2.63, 2.65, 2.45, 3.71, 2.44, 2.13, 3.25,2.83, 1.95, 1.90, 1.75, 1.76, 2.06]
y13=[2.83, 3.36, 3.17, 3.13, 5.33, 4.92, 4.52, 2.71, 2.33, 3.05, 2.64, 2.49, 2.59, 2.79, 2.54, 2.80, 3.31, 2.92]
y14=[0, 0.31, 0.26, 0.22, 0.19, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0]
y15=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.45, 0.89, 1.75, 1.99]
y16=[0, 0, 0, 0, 0, 0.19, 0.17, 0.46, 1.29, 1.36, 1.50, 1.75, 1.91, 1.80, 2.57, 2.46, 3.77, 3.41]
y17=[10.20, 11.93, 10.55, 10.27, 8.81, 11.53, 9.95, 9.66, 9.00, 8.55, 7.76, 7.61, 5.64, 5.50, 6.00, 5.53, 5.16, 4.92]
y18=[1.23, 1.38, 1.53, 1.25, 1.00, 0.91, 1.01, 0.95, 0.86, 0.81, 0.74, 0.76, 1.64, 1.44, 2.38, 2.63, 2.46, 2.38]
y19=[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.1,0.6]
y20=[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,.02,0.18,0.22]
y21=[33.71, 17.25, 12.14, 17.86, 19.16, 21.36, 16.58, 14.26, 11.43, 10.18, 8.76, 6.11, 7.86, 6.94, 5.18, 5.19, 3.77, 3.65]
y22 =[0.39, 0.37, 0.21, 0.21, 0.19, 0.25, 0.23, 0.15, 0.16, 0.14, 0.11, 0.09, 0.07, 0.16, 0.19, 0.28, 0.29, 0.29]

the_colors = sns.cubehelix_palette(22, start=0.5, rot=-.75)

#### 2-1 · 기본값 그대로
`plt.stackplot(x, y1, y2, ...)` — y 를 아래에서부터 차례로 쌓는다.

기본 색은 10 개뿐이라 **22 나라에서 색이 반복된다**. 어느 띠가 어느 나라인지 구별이 안 된다.

**🎯 강의 포인트**
- `stackplot(x, y1, y2, …)` — y 를 **아래부터 차례로 쌓음** (y1 = 콩고가 맨 아래)
- 기본 색은 10 개 → 22 나라에서 **색이 반복** → 어느 띠가 어느 나라인지 모름
- `labels=` 은 이름표만 · 아직 범례 없음


이 단계 코드는 길어서 (`y1` ~ `y22`) 통째로 준다. **실행만** 하고, 다음 단계부터 이 셀을 복사해서 쓴다.

In [ ]:
plt.figure(1, figsize=(5, 5))

plt.stackplot(year, y1, y2, y3, y4, y5, y6, y7, y8, y9, y10, y11,
              y12, y13, y14, y15, y16, y17, y18, y19, y20, y21, y22,
              labels=countries)

plt.show()

#### 2-2 · 22 개의 순차형 색
💡5 `cubehelix` — 밝기가 차례로 변하면서 색상이 돌아서 이웃한 띠끼리 구별되고 흑백에서도 밝기 차이가 남는다.

**🎯 강의 포인트**
- `cubehelix` — 밝기는 한 방향으로 꾸준히 변하고, 색상은 나선처럼 회전
  - 흑백 인쇄에도 순서 유지 · 이웃 띠끼리 색이 달라 구별
- `start` = 시작 색 · `rot` = 회전 바퀴 수 (음수 = 반대 방향, 0 = 단색 그라데이션)
- 영상 7:44 — start 를 바꾸면 색띠가 바뀜, "이것저것 넣어 보라"
- 관심 대상은 DRC 와 잠비아 — 나머지는 배경 (영상 8:10)


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — **맨 위에**
```python
the_colors = sns.cubehelix_palette(22, start=0.5, rot=-.75)   # ← 바꿔 보기 : start=2 · rot=0.5 · sns.color_palette('rocket', 22)
```

✏️ 줄 안에 끼워 넣기 — `⟦ ⟧` 안만 새로 친다
```python
labels=countries⟦, colors=the_colors⟧)
```

In [ ]:
# 2-2 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 2-3 · 범위 · 축 이름
💡7 x 는 데이터 양 끝에 맞추고, y 는 0 ~ 100 % 로 고정. 띠 위의 빈 공간이 사라진다.

**🎯 강의 포인트**
- x 는 데이터 양 끝 (1998 ~ 2015) 에 맞춤 · y 는 0 ~ 100 % 로 고정
- 띠 위의 빈 공간이 사라짐 → 합 100 % 라는 의미가 그림에 드러남


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `labels=countries, colors=the_colors)` **다음 줄에**
```python
plt.xlim([1998, 2015])
plt.ylim([0, 100])
plt.ylabel('market share of Co production %')
plt.xlabel('year')
```

In [ ]:
# 2-3 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 2-4 · 범례 — 기본값
`plt.legend()` 를 그냥 부르면 22 줄 상자가 **그림을 덮는다** (💡10).

**🎯 강의 포인트**
- `legend()` 만 부르면 22 줄 상자가 **데이터를 덮음**
- 원칙: 범례는 데이터를 가리지 않는 곳에


🔮 **예측** — `plt.legend()` 만 부르면 22 줄 범례는 어디에 생길까?

**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `plt.xlabel('year')` **다음 줄에**
```python
plt.legend()
```

In [ ]:
# 2-4 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 2-5 · 범례를 그림 밖으로
`bbox_to_anchor` — 축 안쪽이 0 ~ 1. 1 보다 크면 그림 밖. 22 줄이라 글자도 줄인다.

**🎯 강의 포인트**
- `bbox_to_anchor` — 축 안쪽 = 0 ~ 1 · 1 보다 크면 그림 **밖**
- (1.51, 1.015) 는 상자가 그림 오른쪽 밖에 오도록 **눈으로 맞춘 값**
- 22 줄이라 `fontsize=9` 로 줄임 (영상 10:43)
- 같은 위치를 더 읽기 쉽게: `bbox_to_anchor=(1.02, 1), loc='upper left'`


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

✏️ 줄 안에 끼워 넣기 — `⟦ ⟧` 안만 새로 친다
```python
plt.legend(⟦bbox_to_anchor=(1.51, 1.015⟧)⟦, fontsize=9)⟧
```

In [ ]:
# 2-5 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 2-6 · 범례 순서를 그림과 같게
💡10 그림은 콩고가 **맨 아래**인데 범례는 콩고가 **맨 위** — 거꾸로다.
범례의 손잡이 (색 네모) 와 이름을 꺼내서 뒤집는다.

**🎯 강의 포인트**
- 범례는 그린 순서대로 위 → 아래 · stackplot 은 아래 → 위 → **거꾸로**
- `plt.gca()` = get current axes · 지금 그리는 축
- `get_legend_handles_labels()` → 색 네모 (handles) 와 이름 (labels) 목록을 꺼냄
- `reversed` 로 뒤집어 다시 넘김 → 범례 위아래 = 그림 띠 순서 (영상 10:50)
- matplotlib 3.7 이상: `plt.legend(reverse=True)` 한 줄로도 가능


🔮 **예측** — 범례의 맨 위 나라는 그림의 맨 아래 띠와 같을까?

**이번 단계** (주석 `# …` 은 안 쳐도 된다)

✏️ 바꾸기 — 이 줄(들)을
```python
plt.legend(bbox_to_anchor=(1.51, 1.015), fontsize=9)   # ← 바꿔 보기 : (1.02, 1), loc='upper left'
```

　　　→ 이렇게
```python
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(list(reversed(handles)), list(reversed(labels)),
           bbox_to_anchor=(1.51, 1.015), fontsize=9)
```

In [ ]:
# 2-6 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 2-7 · 그림 위에 직접 쓰기
💡11 중요한 두 나라와 사건을 그림에 바로 쓴다.
- `axvline` 세로선 · `'\n'` 줄바꿈 · 어두운 띠 위에는 흰 글씨 · `rotation=90` 세로 글씨

**🎯 강의 포인트**
- **범례를 찾아보게 하지 말고 그림에 직접 쓴다** — 메시지에 필요한 것만
- `axvline(x=2003)` — 전쟁 종료 시점 세로선 (영상 9:37)
- `'\n'` 줄바꿈 → 긴 이름이 그림 밖으로 안 나가게 (영상 10:05)
- 어두운 띠 위에는 흰 글씨 · `rotation=90` 세로 글씨
- 22 개 중 글자로 쓴 건 **두 나라와 사건 하나뿐**


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `plt.xlabel('year')` **다음 줄에**
```python
plt.axvline(x=2003, linestyle='dotted', color='black')        # 전쟁이 끝난 해
plt.text(2006, 10, 'Democratic Republic \n      of the Congo')   # \n 으로 두 줄
plt.text(1999, 92, 'Zambia', color='white')                    # ← 바꿔 보기 : color='black'
plt.text(2003.5, 15, 'End of Second Congo War', rotation=90)
```

In [ ]:
# 2-7 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 2-8 · 저장
💡13 범례가 그림 밖에 있어도 `bbox_inches='tight'` 가 같이 담는다.

**🎯 강의 포인트**
- 범례가 그림 밖에 있어도 `bbox_inches='tight'` 가 같이 담음
- 영상 11:43 — "범례가 아직 작다, 더 고칠 데가 있다" → 생각할 거리: 작은 나라는 'others' 로 묶으면?


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `bbox_to_anchor=(1.51, 1.015), fontsize=9)` **다음 줄에**
```python
plt.savefig('DRC.png', dpi=300, bbox_inches='tight')
```

In [ ]:
# 2-8 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


In [ ]:
#@title 🏁 체크포인트 · 2 절 완성 코드 (막혔으면 실행)
the_colors = sns.cubehelix_palette(22, start=0.5, rot=-.75)   # ← 바꿔 보기 : start=2 · rot=0.5 · sns.color_palette('rocket', 22)

plt.figure(1, figsize=(5, 5))

plt.stackplot(year, y1, y2, y3, y4, y5, y6, y7, y8, y9, y10, y11,
              y12, y13, y14, y15, y16, y17, y18, y19, y20, y21, y22,
              labels=countries, colors=the_colors)

plt.xlim([1998, 2015])
plt.ylim([0, 100])
plt.ylabel('market share of Co production %')
plt.xlabel('year')

plt.axvline(x=2003, linestyle='dotted', color='black')        # 전쟁이 끝난 해
plt.text(2006, 10, 'Democratic Republic \n      of the Congo')   # \n 으로 두 줄
plt.text(1999, 92, 'Zambia', color='white')                    # ← 바꿔 보기 : color='black'
plt.text(2003.5, 15, 'End of Second Congo War', rotation=90)

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(list(reversed(handles)), list(reversed(labels)),
           bbox_to_anchor=(1.51, 1.015), fontsize=9)

plt.savefig('DRC.png', dpi=300, bbox_inches='tight')

plt.show()

---
## 3 · 다중 패널 

전기변색 (electrochromic) 소재의 투과율. 전압을 걸었을 때 (30 · 60 · 90 kΩ) 와 꺼졌을 때 (clear) — 패널 두 개, 그다음 세 개.

**이 절의 의미**
- 데이터: 전기변색 (electrochromic) 소재 — 전압을 걸면 투과율이 바뀜 (스마트 윈도)
  - 왼쪽: 30 · 60 · 90 kΩ 에서 켠 상태 · 오른쪽: 같은 조건의 clear 상태
- 같은 축 범위로 나란히 → 두 상태의 투과율 차이를 비교하는 그림
- 다중 패널 = Sparks 가 파이썬으로 그리는 **주된 이유 중 하나** (영상 23:22)
- Part I 원칙: 인셋 대신 패널
- 진행: 두 칸 → 문제 찾기 (겹친 숫자 · 중복 이름) → 고치기 → 셋째 칸


In [ ]:
#@title 🔒 데이터 (실행만)
df_full = pd.read_csv(BASE + 'spectroscopy.csv')
print(len(df_full), '줄')

#### 3-1 · 전체 데이터 그대로
점이 너무 촘촘해서 **점 하나하나가 안 보인다**.

**🎯 강의 포인트**
- 점이 너무 촘촘해 **점 하나하나가 안 보임** (영상 13:46)
- 데이터가 많다고 다 그리는 게 좋은 게 아님


**이번 단계 — 새 셀에 직접 친다** (주석 `# …` 은 안 쳐도 된다)
```python
d = df_full
plt.figure(figsize=(5, 5))
plt.plot(d['X 90 Kohm'], d['Y 90 Kohm'], linestyle='none', marker='^', mfc='w', markersize=8)
plt.xlim(400, 800); plt.ylim(60, 120)
plt.show()
```

In [ ]:
# 3-1 · 직접 친다


#### 3-2 · 10 줄에 1 줄만
💡9 `df.iloc[::10, :]` — 10 번째 줄마다 하나. **줄였다는 사실을 그림 설명에 꼭 쓴다** ("1/10 로 표시").

**🎯 강의 포인트**
- `df.iloc[::10, :]` — 10 번째 줄마다 하나 → 1/10 만 남김
- 점이 보이고 그림이 깔끔해짐
- **줄였다는 사실을 그림 설명에 반드시 쓴다** — "데이터를 바꿨으면 바꿨다고 말하라" (영상 14:20)


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

✏️ 줄 안에 끼워 넣기 — `⟦ ⟧` 안만 새로 친다
```python
d = df_full⟦.iloc[::10, :]⟧
```

In [ ]:
# 3-2 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


In [ ]:
#@title 🔒 데이터 (실행만)
# 이제부터 쓸 데이터 — 1/10 로 줄인 것
df = df_full.iloc[::10, :]
x90, y90 = df['X 90 Kohm'].dropna(), df['Y 90 Kohm'].dropna()
x60, y60 = df['X 60 Kohm'].dropna(), df['Y 60 Kohm'].dropna()
x30, y30 = df['X 30 Kohm'].dropna(), df['Y 30 Kohm'].dropna()
x90c, y90c = df['X 90 Kohm-clear'].dropna(), df['Y 90 Kohm-clear'].dropna()
x60c, y60c = df['X 60 Kohm-clear'].dropna(), df['Y 60 Kohm-clear'].dropna()
x30c, y30c = df['X 30 Kohm-clear'].dropna(), df['Y 30 Kohm-clear'].dropna()
xmin, xmax, ymin, ymax = 400, 800, 60, 120
c6 = sns.color_palette('rocket', 6)       # 💡5 6 세트 → rocket 6 색

#### 3-3 · GridSpec 으로 두 칸
그림판을 **4 × 4 격자**로 자르고, 패널마다 몇 칸을 차지할지 정한다. `gs[행, 열]` · 끝 번호는 빠진다.

**🎯 강의 포인트**
- **GridSpec = 그림판을 보이지 않는 격자로 자르고, 패널마다 몇 칸을 차지할지 정함** (엑셀 셀 병합)
- `GridSpec(4, 4)` — 4 행 × 4 열 · `gs[행, 열]` · 끝 번호 미포함
  - 왼쪽 `gs[0:4, 0:2]` · 오른쪽 `gs[0:4, 2:4]`
- `add_subplot` 이 만든 패널이 '현재 축' → 이후 `plt.` 명령은 그 패널로
- `wspace` · `hspace` = 가로 · 세로 간격 (패널 폭 대비 비율)
- 일부러 남긴 문제 찾기: **400 · 800 겹침 · 축 이름 두 번**


**이번 단계 — 새 셀에 직접 친다** (주석 `# …` 은 안 쳐도 된다)
```python
fig = plt.figure(1, figsize=(5, 5))
gs = gridspec.GridSpec(4, 4)               # 4 행 × 4 열
gs.update(wspace=0.2, hspace=0.25)
# 첫 패널 — 왼쪽 절반
fig.add_subplot(gs[0:4, 0:2])              # [행, 열]
plt.plot(x90, y90, linestyle='none', marker='^', mfc='w', markersize=8, color=c6[0], label='90')
plt.plot(x60, y60, linestyle='none', marker='o', mfc='w', markersize=8, color=c6[1], label='60')
plt.plot(x30, y30, linestyle='none', marker='s', mfc='w', markersize=8, color=c6[2], label='30')
plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()
plt.xlabel('Wavelength (nm)'); plt.ylabel(r'Transparency % $\alpha$')
# 둘째 패널 — 오른쪽 절반
fig.add_subplot(gs[0:4, 2:4])
plt.plot(x90c, y90c, linestyle='none', marker='^', mfc='w', markersize=8, color=c6[3], label='90 clear')
plt.plot(x60c, y60c, linestyle='none', marker='o', mfc='w', markersize=8, color=c6[4], label='60 clear')
plt.plot(x30c, y30c, linestyle='none', marker='s', mfc='w', markersize=8, color=c6[5], label='30 clear')
plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()
plt.xlabel('Wavelength (nm)'); plt.ylabel(r'Transparency % $\alpha$')
plt.show()
```

💡 둘째 패널은 **첫 패널 블록을 복사**해서 네 곳만 바꾼다 — `gs[0:4, 2:4]` · 데이터 `x90c …` · 색 `c6[3] c6[4] c6[5]` · label `'… clear'`

In [ ]:
# 3-3 · 새로 친다 (위 셀을 복사하지 않는다)


🔁 **바꿔 보기** — 실행만 해서 비교한다

In [ ]:
# 바꿔 보기 · 패널 사이 간격 wspace 
for w in [1, 0, 0.2]:                      # 아주 넓게 · 붙이기 ·
    fig = plt.figure(figsize=(5, 3))
    gs = gridspec.GridSpec(4, 4); gs.update(wspace=w)
    fig.add_subplot(gs[0:4, 0:2]); plt.plot(x90, y90, '^', mfc='w', color=c6[0]); plt.xlim(xmin, xmax); plt.ylim(ymin, ymax)
    fig.add_subplot(gs[0:4, 2:4]); plt.plot(x90c, y90c, '^', mfc='w', color=c6[3]); plt.xlim(xmin, xmax); plt.ylim(ymin, ymax)
    plt.suptitle(f'wspace={w}', fontsize=12); plt.show()

#### 3-4 · 눈금을 네 변 · 안쪽 + 손으로 정한 간격
💡6 💡7 자동 눈금은 양 끝 (400, 800) 을 다 찍어서 붙은 패널에서 겹친다.
왼쪽은 300 ~ 700 (범위 안에서는 500 · 700), 오른쪽도 500 · 700 만. `701` 인 이유는 끝 값 미포함.

**🎯 강의 포인트**
- 자동 눈금은 양 끝 400 · 800 을 다 찍음 → 붙은 패널에서 **숫자가 겹침** (영상 17:45)
- 눈금 위치를 직접: `np.arange(300, 701, 200)` → 범위 안에서는 500 · 700 만
- 701 = 끝 값 미포함 규칙 (1-10 과 같음)
- 눈금 네 변 · 안쪽은 템플릿 그대로 복사


🔮 **예측** — 자동 눈금 그대로면 두 패널이 만나는 아래쪽에 무엇이 보일까?

**이번 단계** (주석 `# …` 은 안 쳐도 된다)

➕ 추가 — `plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()` (같은 줄이 여럿 — 위에서 1 번째) **다음 줄에**
```python
plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
plt.xticks(np.arange(300, 701, 200))       # ← 바꿔 보기 : 이 줄을 지우면 400 · 800 이 겹친다
plt.yticks(np.arange(60, 121, 20))
```

➕ 추가 — `plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()` (같은 줄이 여럿 — 위에서 2 번째) **다음 줄에**
```python
plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
plt.xticks(np.arange(100, 800, 200))
plt.yticks(np.arange(60, 121, 20))
```

In [ ]:
# 3-4 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 3-5 · 같은 정보는 한 번만
💡8 오른쪽 패널의 세로 숫자 · 세로축 이름을 지운다. 가로축 이름은 `plt.xlabel` 대신 **`plt.text` 로 두 패널 가운데에 하나만** — 위치는 데이터 좌표 (650, 52), 조금씩 옮겨 가며 맞춘다 .

**🎯 강의 포인트**
- **같은 정보는 한 번만** — 오른쪽 세로 숫자 · 세로축 이름 삭제 (영상 21:04)
- 가로축 이름 `xlabel` 은 각 패널 가운데에 붙음 → 두 패널 공통 이름이 안 됨 (영상 19:35)
- 대신 `plt.text` 로 **두 패널 가운데에 하나만** — 좌표 (645, 52) 는 시행착오로 맞춤
- 패널 사이 간격 0.2 ~ 0.25 — 작은 틈이 있어야 두 데이터가 구별됨 (Sparks 의 선택)


**이번 단계** (주석 `# …` 은 안 쳐도 된다)

✏️ 바꾸기 — 이 줄(들)을 (같은 줄이 여럿 — 위에서 1 번째)
```python
plt.xlabel('Wavelength (nm)'); plt.ylabel(r'Transparency % $\alpha$')
```

　　　→ 이렇게
```python
plt.ylabel(r'Transparency % $\alpha$')
plt.text(645, 52, 'Wavelength (nm)')      # ← 바꿔 보기 : 650 · 52 → 50
```

✏️ 바꾸기 — 이 줄(들)을 (같은 줄이 여럿 — 위에서 2 번째)
```python
plt.xlabel('Wavelength (nm)'); plt.ylabel(r'Transparency % $\alpha$')
```

　　　→ 이렇게
```python
plt.tick_params(labelleft=False)           # 세로 숫자 끄기 · ylabel 도 쓰지 않는다
```

In [ ]:
# 3-5 · 위 셀을 복사해 붙여넣고, 안내된 줄을 넣는다


#### 3-6 · 셋째 패널 — 복사해서 칸만 바꾼다
둘째 패널을 통째로 복사해서 오른쪽을 **위 (0:2) · 아래 (2:4)** 로 나눈다.
문제: 위 패널의 가로 숫자가 아래 패널과 부딪히고, 세로 범위가 왼쪽과 같다는 게 한눈에 안 보인다.

**🎯 강의 포인트**
- 둘째 패널 코드를 **통째로 복사** → 칸만 위 (0:2) · 아래 (2:4) 로 (영상 21:38)
- 격자를 4 × 4 로 잘게 잘라 둔 이유 = 칸 번호만 바꿔 패널을 나눌 수 있음
- 새 문제: 위 칸의 가로 숫자가 아래 칸과 부딪힘 · 세로 범위가 왼쪽과 같다는 게 한눈에 안 보임


#### 3-7 · 숫자 정리 — 위 칸 가로 숫자 끄기, 세로 숫자는 오른쪽에
💡8 위 칸의 가로 숫자는 아래 칸과 같으니 끄고, 오른쪽 두 칸은 높이가 달라서 세로 숫자를 **오른쪽 변**에 다시 켠다. 완성.

### 🎯 완성 미션 — 3-6 · 3-7 을 한 번에
아래 셀은 거의 완성된 코드다. `____` 다섯 곳만 채우면 세 칸 그림이 된다.

| 블록 | 하는 일 | 힌트 |
|---|---|---|
| 둘째 패널 | 오른쪽 **위** 칸 | 4 × 4 격자에서 위 두 행 → `gs[?, 2:4]` |
| 셋째 패널 | 오른쪽 **아래** 칸 | 아래 두 행 |
| 둘째 패널 숫자 | 가로 숫자는 아래 칸과 같다 → 끈다 · 세로 숫자는 오른쪽 변에 켠다 | `True` / `False` |
| 셋째 패널 숫자 | 세로 숫자만 오른쪽에 | `True` / `False` |

**🎯 강의 포인트**
- 위 칸 `labelbottom=False` — 아래 칸과 같은 가로 숫자이므로 끔 (영상 22:26)
- 오른쪽 두 칸 `labelright=True` — 높이가 달라 같은 60 ~ 120 인지 안 보이므로 오른쪽 변에 숫자 (영상 22:43)
- 완성 — **격자 크기는 자유**: 리트벨트 그림은 15 × 1 (영상 23:12)
- 정리: 패널 그림의 핵심은 '같은 정보 지우기 · 다른 정보 살리기'


In [ ]:
fig = plt.figure(1, figsize=(5, 5))
gs = gridspec.GridSpec(4, 4)               # 4 행 × 4 열
gs.update(wspace=0.2, hspace=0.25)

# 첫 패널 — 왼쪽 절반
fig.add_subplot(gs[0:4, 0:2])              # [행, 열]
plt.plot(x90, y90, linestyle='none', marker='^', mfc='w', markersize=8, color=c6[0], label='90')
plt.plot(x60, y60, linestyle='none', marker='o', mfc='w', markersize=8, color=c6[1], label='60')
plt.plot(x30, y30, linestyle='none', marker='s', mfc='w', markersize=8, color=c6[2], label='30')
plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()
plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
plt.xticks(np.arange(300, 701, 200))
plt.yticks(np.arange(60, 121, 20))
plt.ylabel(r'Transparency % $\alpha$')
plt.text(645, 52, 'Wavelength (nm)')      # ← 바꿔 보기 : 650 · 52 → 50

# 둘째 패널 — 오른쪽 위
fig.add_subplot(gs[____, 2:4])
plt.plot(x90c, y90c, linestyle='none', marker='^', mfc='w', markersize=8, color=c6[3], label='90 clear')
plt.plot(x60c, y60c, linestyle='none', marker='o', mfc='w', markersize=8, color=c6[4], label='60 clear')
plt.plot(x30c, y30c, linestyle='none', marker='s', mfc='w', markersize=8, color=c6[5], label='30 clear')
plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()
plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
plt.xticks(np.arange(100, 800, 200))
plt.yticks(np.arange(60, 121, 20))
plt.tick_params(labelbottom=____, labelleft=False, labelright=____)   # ← 바꿔 보기 : labelbottom=True

# 셋째 패널 — 오른쪽 아래
fig.add_subplot(gs[____, 2:4])
plt.plot(x90c, y90c, linestyle='none', marker='^', mfc='w', markersize=8, color=c6[3], label='90 clear')
plt.plot(x60c, y60c, linestyle='none', marker='o', mfc='w', markersize=8, color=c6[4], label='60 clear')
plt.plot(x30c, y30c, linestyle='none', marker='s', mfc='w', markersize=8, color=c6[5], label='30 clear')
plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()
plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
plt.xticks(np.arange(100, 800, 200))
plt.yticks(np.arange(60, 121, 20))
plt.tick_params(labelleft=False, labelright=____)

plt.savefig('multipanel.png', dpi=300, bbox_inches='tight')

plt.show()

🔁 **바꿔 보기** — 실행만 해서 비교한다

In [ ]:
#@title 🏁 체크포인트 · 3 절 완성 코드 (막혔으면 실행)
fig = plt.figure(1, figsize=(5, 5))
gs = gridspec.GridSpec(4, 4)               # 4 행 × 4 열
gs.update(wspace=0.2, hspace=0.25)

# 첫 패널 — 왼쪽 절반
fig.add_subplot(gs[0:4, 0:2])              # [행, 열]
plt.plot(x90, y90, linestyle='none', marker='^', mfc='w', markersize=8, color=c6[0], label='90')
plt.plot(x60, y60, linestyle='none', marker='o', mfc='w', markersize=8, color=c6[1], label='60')
plt.plot(x30, y30, linestyle='none', marker='s', mfc='w', markersize=8, color=c6[2], label='30')
plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()
plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
plt.xticks(np.arange(300, 701, 200))
plt.yticks(np.arange(60, 121, 20))
plt.ylabel(r'Transparency % $\alpha$')
plt.text(645, 52, 'Wavelength (nm)')      # ← 바꿔 보기 : 650 · 52 → 50

# 둘째 패널 — 오른쪽 위
fig.add_subplot(gs[0:2, 2:4])
plt.plot(x90c, y90c, linestyle='none', marker='^', mfc='w', markersize=8, color=c6[3], label='90 clear')
plt.plot(x60c, y60c, linestyle='none', marker='o', mfc='w', markersize=8, color=c6[4], label='60 clear')
plt.plot(x30c, y30c, linestyle='none', marker='s', mfc='w', markersize=8, color=c6[5], label='30 clear')
plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()
plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
plt.xticks(np.arange(100, 800, 200))
plt.yticks(np.arange(60, 121, 20))
plt.tick_params(labelbottom=False, labelleft=False, labelright=True)   # ← 바꿔 보기 : labelbottom=True

# 셋째 패널 — 오른쪽 아래
fig.add_subplot(gs[2:4, 2:4])
plt.plot(x90c, y90c, linestyle='none', marker='^', mfc='w', markersize=8, color=c6[3], label='90 clear')
plt.plot(x60c, y60c, linestyle='none', marker='o', mfc='w', markersize=8, color=c6[4], label='60 clear')
plt.plot(x30c, y30c, linestyle='none', marker='s', mfc='w', markersize=8, color=c6[5], label='30 clear')
plt.xlim(xmin, xmax); plt.ylim(ymin, ymax); plt.legend()
plt.minorticks_on()
plt.tick_params(direction='in', which='major', length=10, bottom=True, top=True, left=True, right=True)
plt.tick_params(direction='in', which='minor', length=5,  bottom=True, top=True, left=True, right=True)
plt.xticks(np.arange(100, 800, 200))
plt.yticks(np.arange(60, 121, 20))
plt.tick_params(labelleft=False, labelright=True)

plt.savefig('multipanel.png', dpi=300, bbox_inches='tight')

plt.show()